# Data Pre-processing Experimentation

In [156]:
import nibabel as nib
import numpy as np
from pathlib import Path
import pandas as pd
import re

In [157]:
class DataSubject:
    def __init__(self, cohort, patient_id, t1, t1c, t2, flair, age, sex, label):
        self.cohort = cohort
        self.patient_id = patient_id

        self.t1 = t1
        self.t1c = t1c
        self.t2 = t2
        self.flair = flair

        self.age = age
        self.sex = sex
        self.label = label

    def load(self, sequence):
        """Load one sequence as a numpy array, e.g. subject.load("flair")."""
        path = getattr(self, sequence)
        return np.asarray(nib.load(path).dataobj, dtype=np.float32)

    def load_nifti(self, sequence):
        """Load one sequence as a nibabel image"""
        return nib.load(getattr(self, sequence))

## UCSF Cohort

In [158]:
df = pd.read_csv("../../data/UCSF-PDGM/UCSF-PDGM-metadata_v5.csv")
df = df[["ID", "Sex", "Age at MRI", "IDH"]]
ucsf_df = df[~df['ID'].str.contains(r'FU\d{3}d', regex=True, na=False)]
ucsf_df

,ID,Sex,Age at MRI,IDH
0,UCSF-PDGM-004,M,66,wildtype
1,UCSF-PDGM-005,F,80,wildtype
2,UCSF-PDGM-007,M,70,wildtype
3,UCSF-PDGM-008,M,70,wildtype
4,UCSF-PDGM-009,F,68,wildtype
...,...,...,...,...
496,UCSF-PDGM-537,M,64,wildtype
497,UCSF-PDGM-538,F,41,wildtype
498,UCSF-PDGM-539,M,72,wildtype
499,UCSF-PDGM-540,F,70,IDH1 p.R132H


In [159]:
root = Path("../../data/UCSF-PDGM/PKG - UCSF-PDGM Version 5/UCSF-PDGM-v5/")

SEQUENCES = [
    "T1",
    "T1c",
    "T2",
    "FLAIR"
]

def folder_id(csv_id, width=4):
    num = int(re.search(r"(\d+)$", csv_id).group(1))
    return f"UCSF-PDGM-{num:0{width}d}"

def find_sequence(csv_id, suffix):
    fid = folder_id(csv_id)
    p = root / f"{fid}_nifti" / f"{fid}_{suffix}_bias.nii.gz"
    return p if p.exists() else None

ucsf_df = ucsf_df.copy()
ucsf_df["patient_id"] = ucsf_df["ID"].apply(folder_id)   # canonical 4-digit ID

for seq in SEQUENCES:
    ucsf_df[seq] = ucsf_df["ID"].apply(lambda pid: find_sequence(pid, seq))

complete = ucsf_df.notna().all(axis=1)
print(complete.all())

True


In [160]:
ucsf_subjects = []

for index, row in ucsf_df.iterrows():
    ucsf_subjects.append(
        DataSubject(
            cohort = "UCSF-PDGM",
            patient_id = row["ID"],
            t1 = row["T1"],
            t1c = row["T1c"],
            t2 = row["T2"],
            flair = row["FLAIR"],
            age = row["Age at MRI"],
            sex = row["Sex"],
            label = row["IDH"]
        )
    )

print(len(ucsf_subjects))

495


## UTSW Cohort

In [161]:
df = pd.read_csv("../../data/UTSW-Glioma/UTSW_Glioma_Metadata.tsv", sep="\t")
df = df[["Subject ID", "Sex at birth", "Age at Imaging", "IDH", "T1", "T1GD", "T2", "T2FLAIR"]]

SEQUENCES = ["T1", "T1GD", "T2", "T2FLAIR"]
df = df[df[SEQUENCES].eq("Available").all(axis=1)]
df = df.dropna(subset=["IDH"])
df = df[["Subject ID", "Sex at birth", "Age at Imaging", "IDH"]]
df

,Subject ID,Sex at birth,Age at Imaging,IDH
0,BT0001,F,44,wild type
1,BT0002,M,58,wild type
2,BT0003,F,22,mutated
3,BT0005,M,67,wild type
4,BT0007,M,30,mutated
...,...,...,...,...
620,BT1300,M,59,mutated
621,BT1301,F,20,mutated
622,BT1303,F,42,mutated
623,BT1311,M,50,wild type


In [162]:
root = Path("../../data/UTSW-Glioma/PKG - UTSW-Glioma/UTSW-Glioma/")

SEQUENCES = [
    "t1", "t1ce", "t2", "flair"
]

def find_sequence(patient_id, suffix):
    p = root / patient_id / f"brain_{suffix}.nii.gz"
    return p if p.exists() else None

utsw_df = df.copy()
for seq in SEQUENCES:
    utsw_df[seq] = utsw_df["Subject ID"].apply(lambda pid: find_sequence(pid, seq))

complete = utsw_df.notna().all(axis=1)
print(complete.all())

True


In [163]:
utsw_subjects = []

for index, row in utsw_df.iterrows():
    utsw_subjects.append(
        DataSubject(
            cohort = "UTSW-Glioma",
            patient_id = row["Subject ID"],
            t1 = row["t1"],
            t1c = row["t1ce"],
            t2 = row["t2"],
            flair = row["flair"],
            age = row["Age at Imaging"],
            sex = row["Sex at birth"],
            label = row["IDH"]
        )
    )

print(len(utsw_subjects))

622


## EGD Cohort

In [164]:
df = pd.read_csv("../../data/EGD/egd_data/metadata/subjects.csv")
df = df[["subject_label", "fields/field[0]/field", "demographics/gender", "demographics/age"]]
df = df.rename(columns={"fields/field[0]/field":"IDH", "demographics/gender":"gender", "demographics/age":"age"})
df

,subject_label,IDH,gender,age
0,EGD-0001,-1.0,male,77.0
1,EGD-0002,-1.0,male,48.0
2,EGD-0003,-1.0,female,32.0
3,EGD-0004,1.0,female,45.0
4,EGD-0005,-1.0,female,39.0
...,...,...,...,...
775,EGD-2570,NaN,NaN,NaN
776,EGD-2577,NaN,NaN,NaN
777,EGD-2587,NaN,NaN,NaN
778,EGD-2615,NaN,NaN,NaN


In [165]:
n0 = len(df)

df = df.replace([-1, "-1"], np.nan)
df = df.replace(r"(?i)^\s*unknown\s*$", np.nan, regex=True)
df = df.dropna().reset_index(drop=True)

print(f"{n0} -> {len(df)} rows")

780 -> 424 rows


In [166]:
root = Path("../../data/EGD/egd_data/images/")

SEQUENCES = {
    "T1" : 1,
    "T1GD": 2,
    "T2": 3,
    "FLAIR": 4
}

def find_sequence(patient_id, suffix):
    p = root / patient_id / f"MR_{patient_id}" / f"scan{SEQUENCES[suffix]}_{suffix}_{suffix}" / "NIFTI" / f"{suffix}.nii.gz"
    return p if p.exists() else None

egd_df = df.copy()
for seq in SEQUENCES.keys():
    egd_df[seq] = egd_df["subject_label"].apply(lambda pid: find_sequence(pid, seq))

complete = egd_df.notna().all(axis=1)
print(complete.all())

True


In [167]:
egd_df

,subject_label,IDH,gender,age,T1,T1GD,T2,FLAIR
0,EGD-0004,1.0,female,45.0,../../data/EGD/egd_data/images/EGD-0004/MR_EGD...,../../data/EGD/egd_data/images/EGD-0004/MR_EGD...,../../data/EGD/egd_data/images/EGD-0004/MR_EGD...,../../data/EGD/egd_data/images/EGD-0004/MR_EGD...
1,EGD-0008,1.0,female,60.0,../../data/EGD/egd_data/images/EGD-0008/MR_EGD...,../../data/EGD/egd_data/images/EGD-0008/MR_EGD...,../../data/EGD/egd_data/images/EGD-0008/MR_EGD...,../../data/EGD/egd_data/images/EGD-0008/MR_EGD...
2,EGD-0009,0.0,male,45.0,../../data/EGD/egd_data/images/EGD-0009/MR_EGD...,../../data/EGD/egd_data/images/EGD-0009/MR_EGD...,../../data/EGD/egd_data/images/EGD-0009/MR_EGD...,../../data/EGD/egd_data/images/EGD-0009/MR_EGD...
3,EGD-0011,1.0,female,49.0,../../data/EGD/egd_data/images/EGD-0011/MR_EGD...,../../data/EGD/egd_data/images/EGD-0011/MR_EGD...,../../data/EGD/egd_data/images/EGD-0011/MR_EGD...,../../data/EGD/egd_data/images/EGD-0011/MR_EGD...
4,EGD-0014,0.0,male,84.0,../../data/EGD/egd_data/images/EGD-0014/MR_EGD...,../../data/EGD/egd_data/images/EGD-0014/MR_EGD...,../../data/EGD/egd_data/images/EGD-0014/MR_EGD...,../../data/EGD/egd_data/images/EGD-0014/MR_EGD...
...,...,...,...,...,...,...,...,...
419,EGD-0765,0.0,male,57.0,../../data/EGD/egd_data/images/EGD-0765/MR_EGD...,../../data/EGD/egd_data/images/EGD-0765/MR_EGD...,../../data/EGD/egd_data/images/EGD-0765/MR_EGD...,../../data/EGD/egd_data/images/EGD-0765/MR_EGD...
420,EGD-0768,0.0,male,56.0,../../data/EGD/egd_data/images/EGD-0768/MR_EGD...,../../data/EGD/egd_data/images/EGD-0768/MR_EGD...,../../data/EGD/egd_data/images/EGD-0768/MR_EGD...,../../data/EGD/egd_data/images/EGD-0768/MR_EGD...
421,EGD-0771,1.0,female,38.0,../../data/EGD/egd_data/images/EGD-0771/MR_EGD...,../../data/EGD/egd_data/images/EGD-0771/MR_EGD...,../../data/EGD/egd_data/images/EGD-0771/MR_EGD...,../../data/EGD/egd_data/images/EGD-0771/MR_EGD...
422,EGD-0772,0.0,female,64.0,../../data/EGD/egd_data/images/EGD-0772/MR_EGD...,../../data/EGD/egd_data/images/EGD-0772/MR_EGD...,../../data/EGD/egd_data/images/EGD-0772/MR_EGD...,../../data/EGD/egd_data/images/EGD-0772/MR_EGD...


In [168]:
egd_subjects = []

for index, row in egd_df.iterrows():
    egd_subjects.append(
        DataSubject(
            cohort = "EGD",
            patient_id = row["subject_label"],
            t1 = row["T1"],
            t1c = row["T1GD"],
            t2 = row["T2"],
            flair = row["FLAIR"],
            age = row["age"],
            sex = row["gender"],
            label = row["IDH"]
        )
    )

print(len(egd_subjects))

424


# Initial ML Clinical Classifiers

In [169]:
subjects = ucsf_subjects + utsw_subjects + egd_subjects

def _is_missing(x):
    return x is None or (isinstance(x, float) and np.isnan(x))

def norm_sex(x):
    if _is_missing(x): return np.nan
    s = str(x).strip().lower()
    if s in {"m", "male"}:   return "male"
    if s in {"f", "female"}: return "female"
    return np.nan                      # unknown, "-1", etc.

LABEL_MAP = {
    "wildtype": 0, "wild type": 0, "wild-type": 0, "wt": 0, "0": 0,
    "mutated": 1, "mutant": 1, "mut": 1, "mutated (nos)": 1, "1": 1,
    "idh1 p.r132h": 1, "idh1 p.arg132his": 1,
    "idh1 p.r132c": 1, "idh1 p.r132g": 1, "idh1 p.r132s": 1,
    "idh2 p.r172k": 1, "idh2 p.arg172trp": 1,
}

def norm_label(x):
    """1 = IDH-mutant (IDH1 or IDH2), 0 = IDH-wildtype."""
    if _is_missing(x): return np.nan
    s = str(x).strip().lower()
    if s.endswith(".0"): s = s[:-2]
    if s not in LABEL_MAP:
        print("UNMAPPED LABEL:", repr(x))
        return np.nan
    return LABEL_MAP[s]

# normalise the objects themselves so everything downstream is consistent
for s in subjects:
    s.sex   = norm_sex(s.sex)
    s.label = norm_label(s.label)
    s.age   = pd.to_numeric(s.age, errors="coerce")

df = pd.DataFrame({
    "cohort":     [s.cohort for s in subjects],
    "patient_id": [s.patient_id for s in subjects],
    "age":        [s.age for s in subjects],
    "sex":        [s.sex for s in subjects],
    "label":      [s.label for s in subjects],
})

# sanity checks
print(df.groupby("cohort")[["age", "sex", "label"]].apply(lambda g: g.isna().sum()))
print(df.groupby("cohort")["label"].agg(["count", "mean"]))      # mean = mutant prevalence
print(df.groupby("cohort")["age"].describe()[["min", "50%", "max"]])
print(df["patient_id"].duplicated().sum(), "duplicate IDs")

n0 = len(df)
df = df.dropna(subset=["age", "sex", "label"])
df = df[df["age"].between(0, 110)].reset_index(drop=True)
df["label"] = df["label"].astype(int)
print(f"{n0} -> {len(df)} after dropping missing/invalid")

             age  sex  label
cohort                      
EGD            0    0      0
UCSF-PDGM      0    0      0
UTSW-Glioma    0    0      0
             count      mean
cohort                      
EGD            424  0.349057
UCSF-PDGM      495  0.208081
UTSW-Glioma    622  0.282958
              min   50%   max
cohort                       
EGD          19.0  57.0  86.0
UCSF-PDGM    17.0  59.0  94.0
UTSW-Glioma  18.0  58.0  85.0
0 duplicate IDs
1541 -> 1541 after dropping missing/invalid


In [171]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support

FEATURE_SETS = {"age": ["age"], "sex": ["sex"], "age+sex": ["age", "sex"]}

def make_model(kind, feats):
    parts = []
    if "age" in feats: parts.append(("age", StandardScaler(), ["age"]))
    if "sex" in feats: parts.append(("sex", OneHotEncoder(drop="if_binary"), ["sex"]))
    pre = ColumnTransformer(parts)
    clf = {
        "logreg": LogisticRegression(class_weight="balanced", max_iter=1000),
        "svm_rbf": SVC(kernel="rbf", class_weight="balanced"),
        "gboost": GradientBoostingClassifier(random_state=0),
    }[kind]
    return Pipeline([("pre", pre), ("clf", clf)])

def scores(model, X):
    clf = model.named_steps["clf"]
    return clf.decision_function(model.named_steps["pre"].transform(X)) \
        if hasattr(clf, "decision_function") else model.predict_proba(X)[:, 1]

def run_once(data, feats, kind, seed, test_size=0.25):
    strat = data["cohort"] + "_" + data["label"].astype(str)
    Xtr, Xva, ytr, yva = train_test_split(
        data[feats], data["label"], test_size=test_size,
        stratify=strat, random_state=seed)
    m = make_model(kind, feats).fit(Xtr, ytr)
    pred = m.predict(Xva)
    p, r, f1, _ = precision_recall_fscore_support(
        yva, pred, average="binary", pos_label=1, zero_division=0)
    return dict(auc=roc_auc_score(yva, scores(m, Xva)),
                acc=accuracy_score(yva, pred),
                precision=p, recall=r, f1=f1,
                n_val=len(yva), baseline_acc=1 - yva.mean() if yva.mean() < .5 else yva.mean())

results = []
groups = {c: g for c, g in df.groupby("cohort")}
groups["COMBINED"] = df

for name, data in groups.items():
    for fs_name, feats in FEATURE_SETS.items():
        for kind in ["logreg", "svm_rbf", "gboost"]:
            runs = pd.DataFrame([run_once(data, feats, kind, seed) for seed in range(20)])
            row = runs.mean().add_suffix("_mean").to_dict()
            row.update(runs[["auc", "acc", "f1"]].std().add_suffix("_sd").to_dict())
            results.append(dict(cohort=name, features=fs_name, model=kind, **row))

results = pd.DataFrame(results)
results.round(3).sort_values(["cohort", "features", "model"])

,cohort,features,model,auc_mean,acc_mean,precision_mean,recall_mean,f1_mean,n_val_mean,baseline_acc_mean,auc_sd,acc_sd,f1_sd
29,COMBINED,age,gboost,0.870,0.832,0.740,0.614,0.670,386.0,0.723,0.019,0.012,0.023
27,COMBINED,age,logreg,0.878,0.814,0.629,0.809,0.707,386.0,0.723,0.021,0.018,0.026
28,COMBINED,age,svm_rbf,0.855,0.814,0.634,0.786,0.701,386.0,0.723,0.023,0.019,0.026
35,COMBINED,age+sex,gboost,0.871,0.838,0.747,0.631,0.684,386.0,0.723,0.020,0.012,0.023
33,COMBINED,age+sex,logreg,0.878,0.813,0.630,0.799,0.704,386.0,0.723,0.021,0.019,0.028
34,COMBINED,age+sex,svm_rbf,0.861,0.814,0.634,0.785,0.701,386.0,0.723,0.022,0.022,0.027
32,COMBINED,sex,gboost,0.504,0.723,0.000,0.000,0.000,386.0,0.723,0.030,0.000,0.000
30,COMBINED,sex,logreg,0.504,0.542,0.285,0.419,0.337,386.0,0.723,0.030,0.054,0.021
31,COMBINED,sex,svm_rbf,0.504,0.535,0.286,0.443,0.342,386.0,0.723,0.030,0.076,0.030
2,EGD,age,gboost,0.863,0.810,0.737,0.716,0.723,106.0,0.651,0.032,0.025,0.041


In [178]:
summary = pd.DataFrame({
    "AUC": lr_age["auc_mean"].round(3).astype(str) + " ± " + lr_age["auc_sd"].round(3).astype(str),
    "Accuracy": lr_age["acc_mean"].round(3),
    "Precision": lr_age["precision_mean"].round(3),
    "Recall": lr_age["recall_mean"].round(3),
    "F1": lr_age["f1_mean"].round(3),
    "Majority baseline acc": lr_age["baseline_acc_mean"].round(3),
    "n val": lr_age["n_val_mean"].astype(int),
})
summary

,AUC,Accuracy,Precision,Recall,F1,Majority baseline acc,n val
cohort,,,,,,,
UCSF-PDGM,0.903 ± 0.026,0.848,0.601,0.850,0.702,0.790,124
UTSW-Glioma,0.879 ± 0.022,0.814,0.637,0.800,0.708,0.718,156
EGD,0.876 ± 0.036,0.814,0.697,0.835,0.758,0.651,106
COMBINED,0.878 ± 0.021,0.814,0.629,0.809,0.707,0.723,386
